> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 4 · Notebook 06 — Orders: one model, two brokers

**Sessions:** S10 (Order types & cross-broker mapping) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Round a limit price onto the tick grid without giving money away.
2. Map one canonical order to IB's and Alpaca's fields.
3. Map both brokers' order statuses to one set of states.
4. Use a property test to show the mapping never changes an order's meaning.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()
from decimal import ROUND_CEILING, ROUND_FLOOR

## 1. Rounding a limit price

A price off the tick grid is rejected (IB error 110). But which way to round? Half-up rounding sometimes moves a **buy** limit *up*: you'd pay more than you decided to. Round in your favour: a BUY rounds **down**, a SELL rounds **up**.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def round_limit(price: Decimal, tick: Decimal, side: str) -> Decimal:
    rounding = ROUND_FLOOR if side == "BUY" else ROUND_CEILING
    return (price / tick).quantize(Decimal("1"), rounding=rounding) * tick

cases = [(Decimal("101.237"), Decimal("0.05"), "BUY"), (Decimal("101.237"), Decimal("0.05"), "SELL"),
         (Decimal("4512.30"), Decimal("0.25"), "BUY"), (Decimal("4512.30"), Decimal("0.25"), "SELL"),
         (Decimal("1.08765"), Decimal("0.00005"), "BUY"), (Decimal("231.55"), Decimal("0.01"), "SELL")]
mine = [p.attempt(round_limit, *c) for c in cases]
mine = p.check("round_limit", mine, [p.round_limit(*c) for c in cases])
mine

In [ ]:
rng = np.random.default_rng(0)
raw = [Decimal(str(round(x, 3))) for x in rng.uniform(4000, 5000, 10_000)]    # ES quotes, tick 0.25
tick = Decimal("0.25")
worse = sum(p.half_up(x, tick) > x for x in raw)
print(f"half-up rounding raises a BUY limit above the intended price in {worse:,} of 10,000 cases")
print(f"side-aware rounding: {sum(p.round_limit(x, tick, 'BUY') > x for x in raw)} cases")

## 2. One order, two brokers

Strategies create one canonical `p.Order`; the adapters translate it. The field names and spellings differ:

| Canonical | IB (`ib_async.Order`) | Alpaca (`alpaca-py` request) |
|---|---|---|
| side `BUY` | `action="BUY"` | `side="buy"` |
| qty | `totalQuantity` | `qty` |
| type `STP_LMT` | `orderType="STP LMT"` | `type="stop_limit"` |
| limit | `lmtPrice` | `limit_price` |
| stop | `auxPrice` | `stop_price` |
| trail % | `trailingPercent` | `trail_percent` |
| outside RTH | `outsideRth` | `extended_hours` (DAY limit orders only) |
| client id | `orderRef` | `client_order_id` |

In [ ]:
pd.DataFrame([vars(o) for o in p.SAMPLE_ORDERS]).fillna("")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

Fields that are `None` are left out of the result (the dict comprehension at the end does that).

In [ ]:
IB_TYPES = {"MKT": "MKT", "LMT": "LMT", "STP": "STP", "STP_LMT": "STP LMT", "TRAIL": "TRAIL"}

def to_ib_fields(o: p.Order) -> dict:
    d = {"action": o.side, "totalQuantity": o.qty, "orderType": IB_TYPES[o.type], "lmtPrice": o.limit,
         "auxPrice": o.stop, "trailingPercent": o.trail_pct, "tif": o.tif,
         "outsideRth": o.outside_rth, "orderRef": o.client_id}
    return {k: v for k, v in d.items() if v is not None}

mine = [to_ib_fields(o) for o in p.SAMPLE_ORDERS]
mine = p.check("to_ib_fields", mine, [p.to_ib_fields(o) for o in p.SAMPLE_ORDERS])
pd.DataFrame(mine).fillna("")

The Alpaca side is written for you (`p.to_alpaca_fields`). It also enforces a broker rule: extended hours only for DAY limit orders.

In [ ]:
display(pd.DataFrame([p.to_alpaca_fields(o) for o in p.SAMPLE_ORDERS]).fillna(""))
gtc_night = p.Order("NVDA", "BUY", Decimal("5"), "LMT", limit=Decimal("118.40"), tif="GTC", outside_rth=True)
try:
    p.to_alpaca_fields(gtc_night)
except ValueError as e:
    print("🛑 refused before it reached the broker:", e)

## 3. Statuses into one state machine

Both brokers report order status, with different words. We map them to one set of states (`PENDING_NEW`, `ACCEPTED`, `PARTIALLY_FILLED`, `FILLED`, `PENDING_CANCEL`, `CANCELLED`, `REJECTED`, `EXPIRED`).

IB has one quirk: there is **no partial-fill status**. A `Submitted` order with `filled > 0` is partially filled. `p.IB_STATUS` maps the words; you add the quirk.

In [ ]:
pd.DataFrame({"IB status": list(p.IB_STATUS), "canonical": list(p.IB_STATUS.values())})

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def map_ib_status(status: str, filled: float) -> str:
    s = p.IB_STATUS[status]
    return "PARTIALLY_FILLED" if s == "ACCEPTED" and filled > 0 else s

cases = [("PendingSubmit", 0), ("PreSubmitted", 0), ("Submitted", 0), ("Submitted", 40), ("Filled", 100),
         ("PendingCancel", 40), ("Cancelled", 40), ("Inactive", 0)]
mine = [map_ib_status(s, f) for s, f in cases]
mine = p.check("map_ib_status", mine, [p.map_ib_status(s, f) for s, f in cases])
list(zip(cases, mine))

## 4. A property test: the mapping never changes meaning

Example tests check the cases you thought of. A **property test** (Hypothesis) generates hundreds of orders and checks a rule that must always hold: both brokers get the same side, quantity and prices, and a rounded limit is never worse than the one asked for.

In [ ]:
from hypothesis import given, settings, strategies as st

prices = st.decimals(min_value=Decimal("0.01"), max_value=Decimal("5000"), places=4)
orders = st.builds(p.Order, symbol=st.sampled_from(["SPY", "AAPL"]), side=st.sampled_from(["BUY", "SELL"]),
                   qty=st.integers(1, 10_000).map(Decimal), type=st.just("LMT"), limit=prices)

@settings(max_examples=300, deadline=None)
@given(orders, st.sampled_from([Decimal("0.01"), Decimal("0.05"), Decimal("0.25")]))
def test_mapping_keeps_meaning(o, tick):
    ib, alp = p.to_ib_fields(o), p.to_alpaca_fields(o)
    assert ib["action"].lower() == alp["side"] and ib["totalQuantity"] == alp["qty"] and ib["lmtPrice"] == alp["limit_price"]
    r = p.round_limit(o.limit, tick, o.side)
    assert (r <= o.limit) if o.side == "BUY" else (r >= o.limit)      # never worse than asked
    assert r % tick == 0                                              # on the grid

test_mapping_keeps_meaning()
print("✔ 300 random orders: mapping and rounding keep their meaning")

## Wrap-up

* Round prices in the trader's favour, in `Decimal`, before the order leaves.
* One canonical order and state set; the adapters translate both ways.
* Property tests find the mapping bug you didn't think to write an example for.
* Graded version: `labs/part04/week15_orders` (with real `ib_async` / `alpaca-py` objects and a Hypothesis test).